# 01 — Extract from Source (Azure SQL via Connection Manager)
Reads the active table list from parent metadata and extracts each table
using Lakehouse Federation: `chinook_azure_sql_catalog.chinook.<table>`.

In [0]:
# Widget parameters
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "raw_zone", "Schema Name")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name", "all", "Table Name")
dbutils.widgets.text("run_id", "", "Run ID (auto-generated if blank)")

catalog = dbutils.widgets.get("catalog_name")
run_id_param = dbutils.widgets.get("run_id")

In [0]:
from datetime import datetime
import uuid

# Generate run_id if not provided
run_id = run_id_param if run_id_param else str(uuid.uuid4())[:8]
execution_ts = datetime.now()
print(f"Run ID: {run_id}")
print(f"Execution timestamp: {execution_ts}")

Run ID: 1bb32e4f
Execution timestamp: 2026-03-28 05:25:40.526282


## 1. Read Active Tables from Metadata

In [0]:
df_metadata = spark.sql(f"""
    SELECT table_id, source_catalog, source_schema, source_table, table_name, primary_key_col, load_sequence
    FROM {catalog}.metadata.pipeline_parent_metadata
    WHERE active_flag = 'Y'
    ORDER BY load_sequence
""")

tables_to_extract = df_metadata.collect()
print(f"Tables to extract: {len(tables_to_extract)}")
for row in tables_to_extract:
    print(f"  {row.load_sequence}. {row.source_catalog}.{row.source_schema}.{row.source_table}")

Tables to extract: 11
  1. chinook_azure_sql_catalog.chinook.album
  2. chinook_azure_sql_catalog.chinook.artist
  3. chinook_azure_sql_catalog.chinook.customer
  4. chinook_azure_sql_catalog.chinook.employee
  5. chinook_azure_sql_catalog.chinook.genre
  6. chinook_azure_sql_catalog.chinook.invoice
  7. chinook_azure_sql_catalog.chinook.invoiceline
  8. chinook_azure_sql_catalog.chinook.mediatype
  9. chinook_azure_sql_catalog.chinook.playlist
  10. chinook_azure_sql_catalog.chinook.playlisttrack
  11. chinook_azure_sql_catalog.chinook.track


## 2. Extract Each Table Using Lakehouse Federation

Tables are read via the foreign catalog: `chinook_azure_sql_catalog.chinook.<table>`

In [0]:
extraction_results = []

for row in tables_to_extract:
    start_time = datetime.now()
    table_name = row.table_name
    source_catalog = row.source_catalog
    source_schema = row.source_schema
    source_table = row.source_table

    try:
        # Read using Lakehouse Federation — foreign catalog reference
        source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
        print(f"\nExtracting: {source_fqn}")

        df = spark.read.table(source_fqn)
        source_count = df.count()

        # Register as temp view for downstream use
        view_name = f"extracted_{table_name}"
        df.createOrReplaceTempView(view_name)

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        print(f"  ✓ {table_name}: {source_count} rows extracted ({duration:.1f}s)")

        extraction_results.append({
            "table_name": table_name,
            "source_count": source_count,
            "status": "success",
            "duration": duration,
            "error": None
        })

        # Log to child metrics
        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'extract',
                current_timestamp(), 'success',
                {source_count}, {source_count},
                NULL, NULL, NULL, NULL,
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e).replace("'", "''")[:500]
        print(f"  ✗ {table_name}: FAILED — {str(e)[:200]}")

        extraction_results.append({
            "table_name": table_name,
            "source_count": 0,
            "status": "failed",
            "duration": duration,
            "error": str(e)[:200]
        })

        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'extract',
                current_timestamp(), 'failed',
                0, 0,
                NULL, NULL, NULL, '{error_msg}',
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)


Extracting: chinook_azure_sql_catalog.chinook.album
  ✓ album: 347 rows extracted (5.8s)

Extracting: chinook_azure_sql_catalog.chinook.artist
  ✓ artist: 275 rows extracted (2.6s)

Extracting: chinook_azure_sql_catalog.chinook.customer
  ✓ customer: 59 rows extracted (2.3s)

Extracting: chinook_azure_sql_catalog.chinook.employee
  ✓ employee: 8 rows extracted (2.4s)

Extracting: chinook_azure_sql_catalog.chinook.genre
  ✓ genre: 25 rows extracted (5.1s)

Extracting: chinook_azure_sql_catalog.chinook.invoice
  ✓ invoice: 412 rows extracted (2.3s)

Extracting: chinook_azure_sql_catalog.chinook.invoiceline
  ✓ invoiceline: 2240 rows extracted (2.3s)

Extracting: chinook_azure_sql_catalog.chinook.mediatype
  ✓ mediatype: 5 rows extracted (2.9s)

Extracting: chinook_azure_sql_catalog.chinook.playlist
  ✓ playlist: 18 rows extracted (2.2s)

Extracting: chinook_azure_sql_catalog.chinook.playlisttrack
  ✓ playlisttrack: 8715 rows extracted (2.2s)

Extracting: chinook_azure_sql_catalog.chinoo

## 3. Extraction Summary

In [0]:
import pandas as pd

summary_df = pd.DataFrame(extraction_results)
print(f"\n{'='*60}")
print(f"EXTRACTION SUMMARY — Run ID: {run_id}")
print(f"{'='*60}")
print(f"Total tables: {len(extraction_results)}")
print(f"Successful:   {sum(1 for r in extraction_results if r['status'] == 'success')}")
print(f"Failed:       {sum(1 for r in extraction_results if r['status'] == 'failed')}")
print(f"Total rows:   {sum(r['source_count'] for r in extraction_results):,}")
print(f"{'='*60}")

display(spark.createDataFrame(summary_df))


EXTRACTION SUMMARY — Run ID: 1bb32e4f
Total tables: 11
Successful:   11
Failed:       0
Total rows:   15,607


table_name,source_count,status,duration,error
album,347,success,5.778542,null
artist,275,success,2.646258,null
customer,59,success,2.297358,null
employee,8,success,2.429867,null
genre,25,success,5.063931,null
invoice,412,success,2.319693,null
invoiceline,2240,success,2.292693,null
mediatype,5,success,2.890057,null
playlist,18,success,2.180757,null
playlisttrack,8715,success,2.236046,null


In [0]:
# Store run_id for downstream notebooks
dbutils.jobs.taskValues.set(key="run_id", value=run_id)
dbutils.jobs.taskValues.set(key="execution_ts", value=str(execution_ts))
print(f"Task values set — run_id: {run_id}")

Task values set — run_id: 1bb32e4f
